In [ ]:
import os, shutil, subprocess, sys, zipfile
from pathlib import Path

inputs = Path("/kaggle/input")
roots = list((inputs / "nanowm-code").rglob("pyproject.toml"))
if not roots:
    roots = list(inputs.rglob("pyproject.toml"))
if len(roots) != 1:
    raise RuntimeError(f"Expected exactly one project root, found {len(roots)}: {roots}")
mounted = roots[0].parent
root = Path("/kaggle/working/project")
root.mkdir(parents=True, exist_ok=True)
shutil.copytree(mounted, root, dirs_exist_ok=True)
for archive in mounted.glob("*.zip"):
    with zipfile.ZipFile(archive) as handle:
        handle.extractall(root)
os.chdir(root)
sys.path.insert(0, str(root))
print("project root:", root)

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "diffusers", "lpips"], check=True)

In [ ]:
# Same reasoning as the profiling kernel: verify code health on Kaggle's
# environment (Python 3.12, different pytest) before trusting any number
# measured here. No GPU-hardware gate needed for M0 -- it is CPU-fine and
# does not touch the training budget, but the AE forward pass benefits from
# whatever GPU is available.
subprocess.run([sys.executable, "-m", "pytest", "tests/", "-q"], check=True)

In [ ]:
subprocess.run([
    sys.executable, "scripts/run_m0_ae_ceiling.py",
    "--out", "/kaggle/working/m0_ae_ceiling.json",
], check=True)

In [ ]:
import json
print(json.dumps(json.load(open("/kaggle/working/m0_ae_ceiling.json")), indent=2))